# Absinthe psychedelic analysis using Terpedia

This Colab is the reproducible analysis companion for the critical review. Terpedia is the source of record for the absinthe inventory, chemical identity records, human-target comparator panel, SAIR joins, evidence-qualified receptor edges, and modulation assignments. The notebook uses checked-in Terpedia-derived snapshots for reproducibility and can optionally query the authenticated Terpedia GCP API for a fresh read-only confirmation.

The analysis does not interpret a missing structure or receptor join as biological inactivity, and it does not infer a psychedelic effect from psychoactivity, toxicity, aroma, or compound presence.

## Colab setup

Clone or upload the Terpedia repository so that this notebook can read `absinthe/data/`. In a hosted Colab runtime, set `REPO_ROOT` to the checked-out `absinthe` directory. No API key is stored in this notebook.

In [ ]:
from pathlib import Path
import csv
import json
import pandas as pd
import matplotlib.pyplot as plt

# Change this only if the repository is in another Colab path.
repo_candidates = [Path('/content/Terpedia/absinthe'), Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in repo_candidates if (p / 'data').exists()), Path('/content/Terpedia/absinthe'))
DATA = REPO_ROOT / 'data'
assert DATA.exists(), f'Cannot find Terpedia data directory: {DATA}'
print('Using Terpedia analysis snapshot:', DATA)

## Terpedia provenance and methods

The snapshot was retrieved from the Terpedia GCP knowledge base on 2026-09-03. Structure records came from the `supernatural2` release; receptor targets came from Terpedia-linked human protein records; SAIR coverage tables record exact-structure and target-panel joins. The review's evidence map is a curated interpretation layer over those Terpedia records, not an assertion that every database row is a validated biological interaction.

In [ ]:
def load_csv(name):
    return pd.read_csv(DATA / name, keep_default_na=False)

compounds = load_csv('absinthe-compounds.csv')
modulation = load_csv('psychedelic-modulation-map.csv')
interactome = load_csv('receptor-interactome.csv')
targets = load_csv('human-neural-receptor-panel.csv')
parquet_join = load_csv('sair-19-target-parquet-join.csv')
projection = load_csv('sair-human-panel-coverage.csv')
framework = load_csv('psychedelic-framework.csv')
provenance = json.loads((DATA / 'gcp-kb-refresh-2026-09-03.json').read_text())

assert len(compounds) == 29
assert set(modulation['compound']) == set(compounds['compound'])
assert len(targets) == 19
assert len(interactome) == 8
assert len(parquet_join) == 684
assert len(projection) == 684
print('Terpedia release:', provenance['retrieval_date'])
print('API:', provenance['api'])
print('Inventory:', len(compounds), 'curated compounds')
print('Human-target comparator panel:', len(targets), 'targets')

## Results 1: curated absinthe inventory

The 29-compound Terpedia analysis inventory contains 26 volatile COA entries and three nonvolatile profile entries. The source COA itself lists 27 volatile rows; p-cymene is explicitly excluded from the current receptor analysis pending identity reconciliation.

In [ ]:
inventory_summary = (
    modulation.groupby(['modulation_level', 'modulation_domain'], as_index=False)
    .agg(compounds=('compound', 'count'), members=('compound', lambda x: ', '.join(x)))
)
display(inventory_summary)

plot = modulation.groupby('modulation_level').size().sort_values(ascending=False)
plot.plot(kind='bar', color='#4c78a8', title='Terpedia absinthe inventory by evidence tier')
plt.ylabel('Number of inventory compounds')
plt.xlabel('Modulation evidence level')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## Results 2: psychedelic framework

The framework requires identity resolution, 5-HT2A pharmacology, plausible CNS exposure, controlled phenomenology, and causal validation. A target-panel record or a structure-search result is not itself evidence of interaction.

In [ ]:
display(framework[['criterion', 'operational_question', 'absinthe_application']])

print('Supported direct 5-HT2A edges:', ((interactome['receptor_gene'] == 'HTR2A') & (interactome['evidence_status'] == 'supported')).sum())
print('Unresolved HTR2A tests:', ((interactome['receptor_gene'] == 'HTR2A') & (interactome['evidence_status'] == 'unestablished')).sum())
print('Directly characterized non-5-HT2A edges:', (interactome['evidence_class'].isin(['direct binding and functional electrophysiology', 'direct binding evidence', 'recombinant/cell functional assay'])).sum())

## Results 3: Terpedia receptor interactome and structure joins

The evidence-qualified interactome contains direct thujone–GABA-A and trans-anethole–TRPA1 evidence, a preclinical linalool candidate, and unresolved comparator/HTR2A tests. The 19-target Terpedia human panel produces 684 compound–target rows (29 × 19); all are reported as `no_match` or `no_join_found` in the inspected SAIR releases. This is a release-level annotation result, not proof of receptor inactivity.

In [ ]:
display(interactome[['compound', 'receptor_gene', 'evidence_class', 'evidence_status', 'effect_or_role']])

print('SAIR full-parquet rows:', len(parquet_join))
print('SAIR full-parquet matches:', (parquet_join['join_status'] != 'no_match').sum())
print('SAIR projection joins:', (projection['join_status'] != 'no_join_found').sum())
print('Human target records:', len(targets))

## Optional live Terpedia GCP confirmation

This cell is read-only and is disabled by default. To run it, obtain the Terpedia key through the approved runtime secret workflow and enter it interactively. The key is never printed or saved. A failed route must be reported as an operational limitation, not as a negative chemical result.

In [ ]:
RUN_LIVE_TERpedia_QUERY = False

if RUN_LIVE_TERpedia_QUERY:
    import getpass, requests
    TERPEDIA_KB_URL = 'https://terpedia-knowledge-nanrsdlaoa-uc.a.run.app'
    key = getpass.getpass('Terpedia GCP API key: ')
    response = requests.post(
        TERPEDIA_KB_URL.rstrip('/') + '/v1/tabular/search',
        headers={'Accept': 'application/json', 'Content-Type': 'application/json', 'x-knowledge-key': key},
        json={'source': 'supernatural2', 'query': 'thujone', 'limit': 20},
        timeout=30,
    )
    response.raise_for_status()
    live = response.json()
    print('Live Terpedia query returned successfully; keys:', list(live)[:10])
else:
    print('Live query disabled; using the checked-in Terpedia snapshot.')

## Conclusion

Within the Terpedia evidence map, absinthe is an alcoholic botanical mixture with plausible non-5-HT2A modulation—especially GABA-A inhibitory-tone/excitability effects and TRPA1 sensory signaling—but no supported direct 5-HT2A edge. The data therefore support the classification **psychoactive and potentially toxic, but not established as a classic psychedelic**.